### 导入需要的安装包

In [1]:
import ee
import geemap

### 设置端口信息

In [3]:
geemap.set_proxy(port=7897)
ee.Authenticate()
# ee.Initialize(project='ee-1065531055')
ee.Initialize()


Successfully saved authorization token.


### 加载显示一下基础影像，如果出现地图画面就说明成功了

In [4]:
Map = geemap.Map()
Map.add_basemap("SATELLITE")
Map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(childr…

### 接下来开始深度学习模型的使用

### 1、导入样本数据

In [5]:
import pandas as pd

# pdData = pd.DataFrame(results, columns=feaProperty)

pdData = pd.read_csv(r"F:\training15_100.csv")
pdData.head(2)


,landcover,A00,A01,A02,A03,A04,A05,A06,A07,A08,...,B8_1,B9,B9_1,PC1,PC2,PC3,PC4,PC5,PC6,elevation
0,0,0.071111,0.059116,0.135886,-0.098424,-0.051734,0.108512,-0.075356,0.093564,-0.113741,...,0.3219,0.001220,0.31060,1.574562,0.602168,1.958282,-5.603435,-3.640442,2.979894,1702
1,0,0.035433,0.029773,0.108512,-0.012057,-0.024606,0.147697,-0.130165,0.119093,-0.079723,...,0.3226,0.001473,0.33315,1.315837,0.639654,1.823983,-4.437515,-1.949374,1.731315,1687


In [6]:
import copy
pdData2 = pdData.copy()

className = pdData2['landcover'].unique().tolist()
print("className 包含的类别是：\n", className)

className 包含的类别是：
 [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 14, 10, 11, 12, 13]


In [7]:
# 随机增加一列，用于后续划分训练和测试集
import numpy as np
np.random.seed(42)
pdData2['random'] = np.random.randint(1, 100, pdData2.shape[0])
pdData2.head(2)

,landcover,A00,A01,A02,A03,A04,A05,A06,A07,A08,...,B9,B9_1,PC1,PC2,PC3,PC4,PC5,PC6,elevation,random
0,0,0.071111,0.059116,0.135886,-0.098424,-0.051734,0.108512,-0.075356,0.093564,-0.113741,...,0.001220,0.31060,1.574562,0.602168,1.958282,-5.603435,-3.640442,2.979894,1702,52
1,0,0.035433,0.029773,0.108512,-0.012057,-0.024606,0.147697,-0.130165,0.119093,-0.079723,...,0.001473,0.33315,1.315837,0.639654,1.823983,-4.437515,-1.949374,1.731315,1687,93


In [8]:
train_data = pdData2[pdData2['random'] <= 70]
val_data = pdData2[pdData2['random'] > 70]

featureName = ['A01', 'A02', 'A03', 'A04', 'A05', 'A06', 'A07', 'A08', 'A09', 'A10', 'A11', 'A12', 'A13', 
               'A14', 'A15', 'A16', 'A17', 'A18', 'A20', 'A21', 'A22', 'A24', 'A25', 'A26', 'A27', 'A28',
                'A29', 'A30', 'A31', 'A33', 'A34', 'A36', 'A37', 'A38', 'A39','A41', 
                 'A42', 'A43', 'A44', 'A46', 'A48', 'A49', 'A51', 'A52', 'A53', 'A54', 'A55', 
                 'A56', 'A57', 'A58', 'A62', 'A61', 'B1', 'B10', 'B11', 'B11_1', 'B12', 'B1_1',
                   'B2', 'B2_1', 'B3', 'B3_1', 'B4', 'B4_1', 'B5', 'B5_1', 'B6', 'B6_1', 'B7', 'B7_1', 'B8', 'B8A', 
                   'B8_1', 'B9', 'B9_1', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6', 'elevation']

labelName = ['landcover']

# === 【关键修复1: 计算标准化参数】 ===
feature_stats = pdData[featureName].agg(['mean', 'std'])
mean_dict = feature_stats.loc['mean'].to_dict()
std_dict = feature_stats.loc['std'].to_dict()

# 检查是否有std=0的特征
zero_std = [f for f in featureName if std_dict[f] == 0]
if zero_std:
    print(f"警告: {len(zero_std)}个特征std=0,已修正为1")
    for f in zero_std:
        std_dict[f] = 1.0

# === 【关键修复2: 对训练和验证数据进行标准化】 ===
def standardize_data(data, mean_dict, std_dict, feature_names):
    """标准化数据"""
    data_copy = data.copy()
    for feat in feature_names:
        data_copy[feat] = (data_copy[feat] - mean_dict[feat]) / std_dict[feat]
    return data_copy

train_data_norm = standardize_data(train_data, mean_dict, std_dict, featureName)
val_data_norm = standardize_data(val_data, mean_dict, std_dict, featureName)

def split_train_val_test(pddata, selectedColumn, classColumn):
    data = np.array(pddata[selectedColumn].values.tolist())
    data_label = np.array(pddata[classColumn])
    return data, data_label

trainData, trainData_label = split_train_val_test(train_data_norm, featureName, labelName)
valData, valData_label = split_train_val_test(val_data_norm, featureName, labelName)

print("trainData shape:\n", trainData.shape)
print("trainData_label shape:\n", trainData_label.shape)
print("valData shape:\n", valData.shape)
print("valData_label shape:\n", valData_label.shape)

trainData shape:
 (2171, 82)
trainData_label shape:
 (2171, 1)
valData shape:
 (858, 82)
valData_label shape:
 (858, 1)


In [9]:
### 把样本集、验证集换成tensor格式

import torch

trainData = torch.tensor(trainData, dtype=torch.double)
trainData_label = torch.tensor(trainData_label, dtype=torch.double)
print(f"trainData shape is {trainData.shape}.\n")
print("trainData_label first element:\n",trainData_label[0])

valData = torch.tensor(valData, dtype=torch.double)
valData_label = torch.tensor(valData_label, dtype=torch.double)
print(f"valData shape is {valData.shape}.\n")




trainData shape is torch.Size([2171, 82]).

trainData_label first element:
 tensor([0.], dtype=torch.float64)
valData shape is torch.Size([858, 82]).



In [10]:
### 将数据集转成torch dataset结构并进行组织

import matplotlib.pyplot as plt
import pandas as pd
import ast
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
from torch import nn
import torch.optim as optim
import torch.nn.functional as F
from torch.nn import ReLU, Sigmoid
torch.set_default_dtype(torch.double)

In [11]:
# 使用TensorDataset构建trainDataset和valDataset
trainDataset = TensorDataset(trainData, trainData_label)
valDataset = TensorDataset(valData, valData_label)

# 定义batch_size
# batch_size = 128

# # 创建 DataLoader
# trainLoader = DataLoader(trainDataset, batch_size=batch_size, shuffle=True)
# valLoader = DataLoader(valDataset, batch_size=batch_size, shuffle=False)  # 在验证集上一般不需要打乱数据

import random
import numpy as np
from torch.utils.data import Sampler, DataLoader

class ClassBalancedBatchSampler(Sampler):
    def __init__(self, labels, num_classes, num_samples_per_class):
        self.num_classes = num_classes
        self.num_samples_per_class = num_samples_per_class
        # 为每个类别收集索引并打乱
        self.class_indices = {
            cls: np.where(labels == cls)[0].tolist()
            for cls in range(num_classes)
        }
        for idxs in self.class_indices.values():
            random.shuffle(idxs)
        # 能组成的完整 batch 数量
        self.num_batches = min(
            len(idxs)//num_samples_per_class
            for idxs in self.class_indices.values()
        )

    def __iter__(self):
        for i in range(self.num_batches):
            batch = []
            for cls in range(self.num_classes):
                start = i * self.num_samples_per_class
                end   = start + self.num_samples_per_class
                batch.extend(self.class_indices[cls][start:end])
            random.shuffle(batch)
            yield batch

    def __len__(self):
        return self.num_batches

# —— 在这里提取标签数组 —— 
labels = trainData_label.squeeze().cpu().numpy().astype(int)

# 构造 sampler：15 个类别，每类 2 个样本
sampler = ClassBalancedBatchSampler(labels, num_classes=15, num_samples_per_class=1)

# 使用 sampler 构造 DataLoader（不再传 batch_size、shuffle）
trainLoader = DataLoader(trainDataset, batch_sampler=sampler)

# 验证集仍可用常规方式
valLoader   = DataLoader(valDataset, batch_size=15, shuffle=False)


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from sklearn.metrics import f1_score

class ImprovedDualStreamNet(nn.Module):

    def __init__(self, in_sem=52, in_phy=30, hidden_dim=64, num_classes=15, dropout_rate=0.4):
        super(ImprovedDualStreamNet, self).__init__()
        
        self.hidden_dim = hidden_dim
        self.dropout_rate = dropout_rate
        
        # ==================== 特征编码器 ====================
        self.sem_encoder = nn.Sequential(
            nn.Linear(in_sem, 128),
            nn.LayerNorm(128),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(128, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU()
        )
        
        # 物理流编码器（L8/S2光谱特征）
        self.phy_encoder = nn.Sequential(
            nn.Linear(in_phy, 128),
            nn.LayerNorm(128),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(128, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU()
        )
        
        # ==================== 2. 特征交互模块 ====================
        # 让两个流相互"看到"对方的信息
        self.cross_interaction = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        
        # ==================== 3. 通道级注意力（SE模块思想） ====================
        # 语义流的通道注意力
        self.sem_channel_attn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 4),  # Squeeze
            nn.ReLU(),
            nn.Linear(hidden_dim // 4, hidden_dim),  # Excitation
            nn.Sigmoid()
        )
        
        # 物理流的通道注意力
        self.phy_channel_attn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 4),
            nn.ReLU(),
            nn.Linear(hidden_dim // 4, hidden_dim),
            nn.Sigmoid()
        )
        
        # ==================== 4. 双向门控融合 ====================
        # 基于两个流的联合表示，生成融合门控权重
        self.fusion_gate = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),  # 输出[B, hidden_dim]的逐通道权重
            nn.Sigmoid()
        )
        
        # 全局流权重（用于分析整体贡献度）
        self.global_weight = nn.Sequential(
            nn.Linear(hidden_dim * 2, 32),
            nn.Tanh(),
            nn.Linear(32, 2),  # 输出2个权重：[sem_weight, phy_weight]
            nn.Softmax(dim=1)
        )
        
        # ==================== 5. 分类器 ====================
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 32),
            nn.LayerNorm(32),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, num_classes)
        )
        
        # 初始化权重
        self._init_weights()
    
    def _init_weights(self):
        """Xavier初始化"""
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.LayerNorm):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
    
    def forward(self, x_sem, x_phy):
        """
        前向传播
        """
        
        # -------- 特征编码 --------
        f_sem = self.sem_encoder(x_sem) 
        f_phy = self.phy_encoder(x_phy) 
        # -------- 特征交互 --------
        combined = torch.cat([f_sem, f_phy], dim=1)  
        interaction = self.cross_interaction(combined) 
        
        # 残差连接：原始特征 + 交互增强
        f_sem_enhanced = f_sem + 0.1 * interaction 
        f_phy_enhanced = f_phy + 0.1 * interaction
        
        # -------- 通道级注意力（SE） --------

        sem_attn = self.sem_channel_attn(f_sem_enhanced) 
        phy_attn = self.phy_channel_attn(f_phy_enhanced) 
        
        f_sem_refined = f_sem_enhanced * sem_attn  # 逐元素相乘
        f_phy_refined = f_phy_enhanced * phy_attn
        
        # -------- 双向门控融合 --------
        combined_refined = torch.cat([f_sem_refined, f_phy_refined], dim=1) 
        
        gate = self.fusion_gate(combined_refined) 
        
        # 融合：gate控制语义流的贡献，(1-gate)控制物理流
        f_fused = gate * f_sem_refined + (1 - gate) * f_phy_refined  
        
        global_weights = self.global_weight(combined_refined)  
        
        output = self.classifier(f_fused)  # [B, num_classes]
        
        # 返回注意力信息用于分析
        attention_info = {
            'channel_gate': gate,     
            'global_weights': global_weights, 
            'sem_channel_attn': sem_attn,     
            'phy_channel_attn': phy_attn    
        }
        
        return output, attention_info


# ==================== 训练函数 ====================

def train_and_validate_improved(model, train_loader, val_loader, criterion, optimizer, 
                                 scheduler, device, epochs=100, split_idx=64):

    metrics_list = []
    best_val_f1 = 0.0
    best_model_state = None
    
    print("=" * 70)
    print(f"开始训练改进版双流网络")
    print(f"特征分割: 前{split_idx}维(语义/AE) | 后{94-split_idx}维(物理/L8+S2)")
    print("=" * 70)

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct_train = 0
        total_train = 0
        all_train_preds = []
        all_train_labels = []
        
        epoch_global_weights = []

        for inputs, labels in train_loader:
            inputs = inputs.to(device)
            labels = labels.to(device).long().squeeze(-1)
            
            # 分割双流输入
            x_sem = inputs[:, :split_idx]
            x_phy = inputs[:, split_idx:]
            
            optimizer.zero_grad()
            outputs, attn_info = model(x_sem, x_phy)
            
            loss = criterion(outputs, labels)
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)

            all_train_preds.extend(predicted.cpu().numpy())
            all_train_labels.extend(labels.cpu().numpy())
            total_train += labels.size(0)
            correct_train += (predicted == labels).sum().item()
            
            epoch_global_weights.append(attn_info['global_weights'].detach().cpu())

        train_acc = 100 * correct_train / total_train
        train_loss = running_loss / len(train_loader)
        train_f1 = f1_score(all_train_labels, all_train_preds, average='macro')
        
        avg_global_weights = torch.cat(epoch_global_weights, dim=0).mean(dim=0)

        # ==================== 验证阶段 ====================
        model.eval()
        correct_val = 0
        total_val = 0
        all_val_preds = []
        all_val_labels = []
        
        val_global_weights = []
        val_channel_gates = []

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs = inputs.to(device)
                labels = labels.to(device).long().squeeze(-1)
                
                x_sem = inputs[:, :split_idx]
                x_phy = inputs[:, split_idx:]
                
                outputs, attn_info = model(x_sem, x_phy)
                
                _, predicted = torch.max(outputs.data, 1)
                
                all_val_preds.extend(predicted.cpu().numpy())
                all_val_labels.extend(labels.cpu().numpy())
                total_val += labels.size(0)
                correct_val += (predicted == labels).sum().item()
                
                # 最后一个epoch保存详细信息
                if epoch == epochs - 1:
                    val_global_weights.append(attn_info['global_weights'].cpu())
                    val_channel_gates.append(attn_info['channel_gate'].cpu())

        val_acc = 100 * correct_val / total_val
        val_f1 = f1_score(all_val_labels, all_val_preds, average='macro')

        scheduler.step()

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        print(f'Epoch [{epoch + 1:3d}/{epochs}] | Loss: {train_loss:.4f} | '
              f'Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}% | '
              f'Val F1: {val_f1:.4f} | '
              f'Sem:{avg_global_weights[0]:.3f} Phy:{avg_global_weights[1]:.3f}')
        
        metrics_list.append({
            'Epoch': epoch + 1,
            'Train Loss': train_loss,
            'Train Acc': train_acc,
            'Val Acc': val_acc,
            'Val F1': val_f1,
            'Sem_Weight': avg_global_weights[0].item(),
            'Phy_Weight': avg_global_weights[1].item()
        })

    # ==================== 保存注意力分析结果 ====================
    if len(val_global_weights) > 0:
        val_global_weights = torch.cat(val_global_weights, dim=0).numpy()
        val_channel_gates = torch.cat(val_channel_gates, dim=0).numpy()
        
        df_global = pd.DataFrame({
            'True_Label': all_val_labels,
            'Predicted': all_val_preds,
            'Weight_Semantic': val_global_weights[:, 0],
            'Weight_Physical': val_global_weights[:, 1]
        })
        
        save_path_global = r"C:\Users\10655\Desktop\test\Attention_Global_Improved.csv"
        df_global.to_csv(save_path_global, index=False)
        print(f"\n[保存] 全局注意力权重: {save_path_global}")
        
        df_channel = pd.DataFrame(val_channel_gates)
        df_channel.columns = [f'Gate_Ch{i}' for i in range(val_channel_gates.shape[1])]
        df_channel['True_Label'] = all_val_labels
        df_channel['Predicted'] = all_val_preds
        
        save_path_channel = r"C:\Users\10655\Desktop\test\Attention_Channel_Improved.csv"
        df_channel.to_csv(save_path_channel, index=False)
        print(f"[保存] 通道级门控权重: {save_path_channel}")

        print("\n" + "=" * 50)
        print("各类别的平均注意力权重分析:")
        print("=" * 50)
        for cls in sorted(set(all_val_labels)):
            mask = [l == cls for l in all_val_labels]
            sem_w = val_global_weights[mask, 0].mean()
            phy_w = val_global_weights[mask, 1].mean()
            dominant = "语义流(AE)" if sem_w > phy_w else "物理流(L8+S2)"
            print(f"类别 {cls:2d}: 语义={sem_w:.3f}, 物理={phy_w:.3f} → 主导: {dominant}")

    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        print(f"\n[信息] 已恢复最佳模型 (Val F1: {best_val_f1:.4f})")

    return metrics_list


# ==================== 运行训练 ====================

# 设备设置
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using Device: {device}")

# 初始化改进模型
model = ImprovedDualStreamNet(
    in_sem=52,    
    in_phy=30,      
    hidden_dim=64,    
    num_classes=15,   
    dropout_rate=0.4  
).to(device)

model.double() 

# 优化器和调度器 
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=30, gamma=0.1)

history = train_and_validate_improved(
    model, trainLoader, valLoader, 
    criterion, optimizer, scheduler, 
    device, epochs=100, split_idx=64
)

Using Device: cuda
开始训练改进版双流网络
特征分割: 前52维(语义/AE) | 后42维(物理/L8+S2)
Epoch [  1/100] | Loss: 2.5327 | Train Acc: 17.45% | Val Acc: 45.57% | Val F1: 0.4209 | Sem:0.467 Phy:0.533
Epoch [  2/100] | Loss: 1.7665 | Train Acc: 42.03% | Val Acc: 70.98% | Val F1: 0.6918 | Sem:0.474 Phy:0.526
Epoch [  3/100] | Loss: 1.3051 | Train Acc: 56.01% | Val Acc: 80.89% | Val F1: 0.8107 | Sem:0.505 Phy:0.495
Epoch [  4/100] | Loss: 1.0402 | Train Acc: 64.18% | Val Acc: 84.27% | Val F1: 0.8413 | Sem:0.514 Phy:0.486
Epoch [  5/100] | Loss: 0.8735 | Train Acc: 70.26% | Val Acc: 87.53% | Val F1: 0.8783 | Sem:0.523 Phy:0.477
Epoch [  6/100] | Loss: 0.7671 | Train Acc: 75.16% | Val Acc: 89.04% | Val F1: 0.8896 | Sem:0.529 Phy:0.471
Epoch [  7/100] | Loss: 0.6249 | Train Acc: 79.08% | Val Acc: 90.56% | Val F1: 0.9062 | Sem:0.529 Phy:0.471
Epoch [  8/100] | Loss: 0.5673 | Train Acc: 81.63% | Val Acc: 92.07% | Val F1: 0.9219 | Sem:0.525 Phy:0.475
Epoch [  9/100] | Loss: 0.5094 | Train Acc: 82.75% | Val Acc: 93.59% |

In [ ]:

model_weights = {
    'sem_encoder': [],    
    'phy_encoder': [],   
    'cross_interaction': [], 
    'sem_channel_attn': [],   
    'phy_channel_attn': [],   
    'fusion_gate': [],    
    'global_weight': [],   
    'classifier': []       
}

# 提取 Linear 层: 索引 0, 4
for layer_idx in [0, 4]:
    weight = model.sem_encoder[layer_idx].weight.detach().cpu().numpy().tolist()
    bias = model.sem_encoder[layer_idx].bias.detach().cpu().numpy().tolist()
    model_weights['sem_encoder'].append([weight, bias])

# 物理流编码器: 结构相同，索引 0, 4
for layer_idx in [0, 4]:
    weight = model.phy_encoder[layer_idx].weight.detach().cpu().numpy().tolist()
    bias = model.phy_encoder[layer_idx].bias.detach().cpu().numpy().tolist()
    model_weights['phy_encoder'].append([weight, bias])

# 交叉交互层: Linear(0) -> ReLU(1) -> Linear(2)
for layer_idx in [0, 2]:
    weight = model.cross_interaction[layer_idx].weight.detach().cpu().numpy().tolist()
    bias = model.cross_interaction[layer_idx].bias.detach().cpu().numpy().tolist()
    model_weights['cross_interaction'].append([weight, bias])

# 语义通道注意力: Linear(0) -> ReLU(1) -> Linear(2) -> Sigmoid(3)
for layer_idx in [0, 2]:
    weight = model.sem_channel_attn[layer_idx].weight.detach().cpu().numpy().tolist()
    bias = model.sem_channel_attn[layer_idx].bias.detach().cpu().numpy().tolist()
    model_weights['sem_channel_attn'].append([weight, bias])

# 物理通道注意力: 结构相同
for layer_idx in [0, 2]:
    weight = model.phy_channel_attn[layer_idx].weight.detach().cpu().numpy().tolist()
    bias = model.phy_channel_attn[layer_idx].bias.detach().cpu().numpy().tolist()
    model_weights['phy_channel_attn'].append([weight, bias])

# 融合门控: Linear(0) -> Tanh(1) -> Linear(2) -> Sigmoid(3)
for layer_idx in [0, 2]:
    weight = model.fusion_gate[layer_idx].weight.detach().cpu().numpy().tolist()
    bias = model.fusion_gate[layer_idx].bias.detach().cpu().numpy().tolist()
    model_weights['fusion_gate'].append([weight, bias])

# 全局权重（可选，GEE中可不实现）: Linear(0) -> Tanh(1) -> Linear(2) -> Softmax(3)
for layer_idx in [0, 2]:
    weight = model.global_weight[layer_idx].weight.detach().cpu().numpy().tolist()
    bias = model.global_weight[layer_idx].bias.detach().cpu().numpy().tolist()
    model_weights['global_weight'].append([weight, bias])

# 分类器: Linear(0) -> LayerNorm(1) -> ReLU(2) -> Dropout(3) -> Linear(4) -> ReLU(5) -> Linear(6)
for layer_idx in [0, 4, 6]:
    weight = model.classifier[layer_idx].weight.detach().cpu().numpy().tolist()
    bias = model.classifier[layer_idx].bias.detach().cpu().numpy().tolist()
    model_weights['classifier'].append([weight, bias])

# 验证提取结果
print("=" * 50)
print("模型权重提取完成!")
print("=" * 50)
for key, value in model_weights.items():
    print(f"{key}: {len(value)} 层")

模型权重提取完成!
sem_encoder: 2 层
phy_encoder: 2 层
cross_interaction: 2 层
sem_channel_attn: 2 层
phy_channel_attn: 2 层
fusion_gate: 2 层
global_weight: 2 层
classifier: 3 层


In [ ]:
mean_list = [mean_dict[f] for f in featureName]
std_list = [std_dict[f] for f in featureName]

print("标准化参数准备完成!")
print(f"Mean前5个值: {mean_list[:5]}")
print(f"Std前5个值: {std_list[:5]}")

标准化参数准备完成!
Mean前5个值: [0.07683158143796633, -0.02957300253067019, 0.0868759192118851, -0.033288545771277656, -0.05312667238653021]
Std前5个值: [0.04172392000129288, 0.06164672694915061, 0.10765541738821595, 0.07126487312252941, 0.04669388848835719]


In [ ]:
roi = ee.FeatureCollection('projects/ee-1065531055/assets/paper4_yumeng_roi')
Map.addLayer(roi, {'color':'blue'}, 'studyArea', shown=True)
Map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(childr…

In [ ]:
dataset = ee.ImageCollection('ESA/WorldCover/v100').first()
dataset1 = dataset.clip(roi)

water_ESA = dataset1.eq(80)
urban_ESA = dataset1.eq(50)
foret_ESA = dataset1.eq(10)

keepMask = water_ESA.Or(urban_ESA).Or(foret_ESA).Not()


# 2. Sentinel-2 云掩膜函数
def rmS2cloud(image):
    qa = image.select('QA60')
    cloudBitMask  = 1 << 10
    cirrusBitMask = 1 << 11
    mask = qa.bitwiseAnd(cloudBitMask).eq(0).And(
           qa.bitwiseAnd(cirrusBitMask).eq(0))
    return image.updateMask(mask) \
                .toDouble().divide(10000) \
                .copyProperties(image, ['system:time_start','system:time_end'])

# 3. 构造 Sentinel-2 合成影像
LL_clipped = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
      .filterBounds(roi)
      .filterDate('2021-06-01', '2021-10-30')
      .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
      .select(['B1','B2','B3','B4','B5','B6','B7','B8','B8A','B9','B11','B12','QA60'])
      .map(rmS2cloud)
      .median()
      .clip(roi)
     )

visParam = {
  "min": 0.0,
  "max": 0.3,
  "bands": ['B4', 'B3', 'B2'],
}

Map.addLayer(LL_clipped, visParam, 'imgS2')
Map


Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(childr…

In [ ]:
sev1 = ee.ImageCollection('GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL') \
    .filterBounds(roi) \
    .filterDate('2021-01-01', '2021-12-31') \
    .median() \
    .clip(roi)

# 2. 设置可视化参数 (Python字典的 Key 需要加引号)
visParams = {
    'min': -0.3, 
    'max': 0.3, 
    'bands': ['A01', 'A16', 'A09']
}

# 3. 添加图层
Map.addLayer(sev1, visParams, 'embeddings')
Map.centerObject(roi)
Map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(childr…

In [ ]:
srtm = ee.Image('USGS/SRTMGL1_003').clip(roi)

# dem10m = srtm.resample('bicubic').reproject(
#     crs=LL_clipped.projection(),  # 这里的 crs 没有引号
#     scale=10
# )

# 3. 计算地形因子 (坡度、坡向、山影等)
terrain = ee.Algorithms.Terrain(srtm)

# # 4. 计算粗糙度 (邻域标准差)
# roughness = dem10m.reduceNeighborhood({
#     'reducer': ee.Reducer.stdDev(),
#     'kernel': ee.Kernel.square(1)
# }).rename('rugosity')

In [ ]:
# 1. 定义去云函数（TOA 影像）
def rmCloud_TOA(image):
    cloud_score = ee.Algorithms.Landsat.simpleCloudScore(image)
    mask = cloud_score.select('cloud').lte(30)
    return image.updateMask(mask)

l8SR = ee.ImageCollection('LANDSAT/LC08/C02/T1_TOA') \
    .filterBounds(roi) \
    .filterDate('2021-06-01', '2021-10-30') \
    .map(rmCloud_TOA) \
    .select(['B1','B2','B3','B4','B5','B6','B7','B8','B9','B10','B11']) \
    .median() \
    .clip(roi)

# 3. 设置可视化参数
rgbVis = {
    'min': 0.0,
    'max': 0.3,
    'gamma': 1.4,
    'bands': ['B4', 'B3', 'B2']
}

Map.addLayer(l8SR, rgbVis, 'l8SR', False)
Map


Map(center=[40.71427804680794, 96.54887517123959], controls=(WidgetControl(options=['position', 'transparent_b…

In [ ]:
bands1=["B2","B3","B4","B8","B11","B12"]

# 定义一个函数，用于对矢量进行矩形分块
def shpRectBlock(vector, numHorizontal, numVertical):
    if not isinstance(vector, (ee.Geometry, ee.Feature, ee.FeatureCollection)):
        raise ValueError('Input must be a Geometry, Feature, or FeatureCollection.')
    if isinstance(vector, ee.FeatureCollection):
        geom = vector.geometry()
    elif isinstance(vector, ee.Feature):
        geom = vector.geometry()
    else:
        geom = vector
    bounds = geom.bounds()
    coords = bounds.coordinates().get(0)
    xmin = ee.Number(coords.get(0).get(0))
    ymin = ee.Number(coords.get(0).get(1))
    xmax = ee.Number(coords.get(2).get(0))
    ymax = ee.Number(coords.get(2).get(1))
    xStep = xmax.subtract(xmin).divide(numHorizontal)
    yStep = ymax.subtract(ymin).divide(numVertical)
    rects = (
        ee.List.sequence(0, numHorizontal - 1)
          .map(lambda i: ee.List.sequence(0, numVertical - 1)
            .map(lambda j: ee.Feature(
                ee.Geometry.Rectangle([
                    xmin.add(xStep.multiply(i)),
                    ymin.add(yStep.multiply(j)),
                    xmin.add(xStep.multiply(i)).add(xStep),
                    ymin.add(yStep.multiply(j)).add(yStep)
                ]).intersection(geom, 1)
            ))
          )
          .flatten()
    )
    return ee.FeatureCollection(rects).filterBounds(geom)

# 定义 PCA 函数
def pca(img):
    bandNames = bands1
    scale = 200
    region = roi

    # 1. 计算均值并中心化
    meanDict = img.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=region,
        scale=scale,
        maxPixels=1e13
    )
    means = ee.Image.constant(meanDict.values(bandNames))
    centered = img.subtract(means)

    # 辅助：生成 PC 波段名（PC1…PC7）
    def getBandNames(prefix):
        return ee.List.sequence(1, len(bandNames)) \
                 .map(lambda n: ee.String(prefix).cat(
                     ee.Number(n).format('%.0f')
                 ))

    # 2. 计算协方差并做特征分解
    arrays = centered.toArray()
    weights = ee.Array([1, 1, 1, 1, 2, 2])
    arrays = arrays.multiply(weights)

    covar = arrays.reduceRegion(
        reducer=ee.Reducer.centeredCovariance(),
        geometry=region,
        scale=scale,
        bestEffort=True,
        maxPixels=1e13
    )
    covarArr = ee.Array(covar.get('array'))
    eigens = covarArr.eigen()
    eigenValues  = eigens.slice(1, 0, 1)
    eigenVectors = eigens.slice(1, 1)

    # 3. 投影到主成分空间并标准化
    arrImg = arrays.toArray(1)
    pcArr  = ee.Image(eigenVectors).matrixMultiply(arrImg)

    sdImg = ee.Image(eigenValues.sqrt()) \
        .arrayProject([0]) \
        .arrayFlatten([getBandNames('SD')])

    pcImage = (
        pcArr
        .arrayProject([0])
        .arrayFlatten([getBandNames('PC')])
        .divide(sdImg)
        .set('eigenValues', eigenValues.toList().flatten())
        .set('eigenVectors', eigenVectors)
    )
    return pcImage

# 应用 PCA 并可视化
pcImage = pca(LL_clipped.select(bands1))

Map = geemap.Map(center=[0, 0], zoom=6)
Map.addLayer(
    pcImage,
    {'bands': ['PC3', 'PC2', 'PC1'], 'min': -2, 'max': 2},
    'Landsat 8 - PCA'
)

Map.addLayerControl()
Map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(childr…

In [ ]:
LL_clipped2 = ee.Image.cat([
    l8SR,
    LL_clipped,
    terrain,
    pcImage,
    sev1
]).toFloat().clip(roi)

# 2. 在主体影像后面添加 VV/VH 纹理波段
LL_clipped2 = LL_clipped2.unmask(0)
LL_clipped1 = LL_clipped2.updateMask(keepMask)

In [ ]:
def dnnLinear(inputArr, weight, bias, activate=None):
    """
    单层线性变换 + 可选激活函数
    
    Args:
        inputArr: 输入数组 [n, 1]
        weight: 权重矩阵
        bias: 偏置向量
        activate: 激活函数类型 ('relu', 'tanh', 'sigmoid', None)
    """
    weight_img = ee.Image(ee.Array(weight))
    bias_img = ee.Image(ee.Array(bias)).toArray(1)
    output = weight_img.matrixMultiply(inputArr).add(bias_img)
    
    if activate == 'relu':
        return output.multiply(output.gt(0))
    elif activate == 'tanh':
        exp_2x = output.multiply(2).exp()
        return exp_2x.subtract(1).divide(exp_2x.add(1))
    elif activate == 'sigmoid':
        return output.multiply(-1).exp().add(1).pow(-1)
    else:
        return output


def standardizeImage(img, mean_list, std_list, band_names):
    """Z-score 标准化"""
    standardized_bands = []
    for i, band_name in enumerate(band_names):
        band = img.select(band_name).toFloat()
        mean_val = float(mean_list[i])
        std_val = float(std_list[i])
        if std_val == 0:
            std_val = 1.0
        band_norm = band.subtract(mean_val).divide(std_val)
        standardized_bands.append(band_norm.rename(band_name))
    return ee.Image.cat(standardized_bands)


def improvedDualStreamInference(img_standardized, model_weights, split_idx=64, n_features=94):
    
    img_array_1d = img_standardized.toArray()  # [94]
    
    # 分割双流
    sem_input = img_array_1d.arraySlice(0, 0, split_idx)  # [64]
    phy_input = img_array_1d.arraySlice(0, split_idx, n_features)  # [30]
    
    img_sem = sem_input.toArray(1)  # [64, 1]
    img_phy = phy_input.toArray(1)  # [30, 1]
    
    w0, b0 = model_weights['sem_encoder'][0]
    f_sem = dnnLinear(img_sem, w0, b0, activate='relu')  # [128, 1]
    
    w1, b1 = model_weights['sem_encoder'][1]
    f_sem = dnnLinear(f_sem, w1, b1, activate='relu')  # [64, 1]
    
    w0, b0 = model_weights['phy_encoder'][0]
    f_phy = dnnLinear(img_phy, w0, b0, activate='relu')  # [128, 1]
    
    w1, b1 = model_weights['phy_encoder'][1]
    f_phy = dnnLinear(f_phy, w1, b1, activate='relu')  # [64, 1]
    
    # 拼接
    f_sem_1d = f_sem.arrayProject([0])  # [64]
    f_phy_1d = f_phy.arrayProject([0])  # [64]
    combined_1d = f_sem_1d.arrayCat(f_phy_1d, 0)  # [128]
    combined = combined_1d.toArray(1)  # [128, 1]
    
    # 交叉交互层
    w0, b0 = model_weights['cross_interaction'][0]
    interaction = dnnLinear(combined, w0, b0, activate='relu')  # [64, 1]
    
    w1, b1 = model_weights['cross_interaction'][1]
    interaction = dnnLinear(interaction, w1, b1, activate=None)  # [64, 1]
    
    interaction_1d = interaction.arrayProject([0])  # [64]
    f_sem_enhanced_1d = f_sem_1d.add(interaction_1d.multiply(0.1))  # [64]
    f_phy_enhanced_1d = f_phy_1d.add(interaction_1d.multiply(0.1))  # [64]
    
    f_sem_enhanced = f_sem_enhanced_1d.toArray(1)  # [64, 1]
    
    w0, b0 = model_weights['sem_channel_attn'][0]
    sem_attn = dnnLinear(f_sem_enhanced, w0, b0, activate='relu')  # [16, 1]
    
    w1, b1 = model_weights['sem_channel_attn'][1]
    sem_attn = dnnLinear(sem_attn, w1, b1, activate='sigmoid')  # [64, 1]
    
    # 物理流通道注意力
    f_phy_enhanced = f_phy_enhanced_1d.toArray(1)  # [64, 1]
    
    w0, b0 = model_weights['phy_channel_attn'][0]
    phy_attn = dnnLinear(f_phy_enhanced, w0, b0, activate='relu')  # [16, 1]
    
    w1, b1 = model_weights['phy_channel_attn'][1]
    phy_attn = dnnLinear(phy_attn, w1, b1, activate='sigmoid')  # [64, 1]
    
    # 应用通道注意力 (逐元素相乘)
    sem_attn_1d = sem_attn.arrayProject([0])  # [64]
    phy_attn_1d = phy_attn.arrayProject([0])  # [64]
    
    f_sem_refined_1d = f_sem_enhanced_1d.multiply(sem_attn_1d)  # [64]
    f_phy_refined_1d = f_phy_enhanced_1d.multiply(phy_attn_1d)  # [64]
    
    combined_refined_1d = f_sem_refined_1d.arrayCat(f_phy_refined_1d, 0)  # [128]
    combined_refined = combined_refined_1d.toArray(1)  # [128, 1]
    
    # 融合门控网络
    w0, b0 = model_weights['fusion_gate'][0]
    gate = dnnLinear(combined_refined, w0, b0, activate='tanh')  # [64, 1]
    
    w1, b1 = model_weights['fusion_gate'][1]
    gate = dnnLinear(gate, w1, b1, activate='sigmoid')  # [64, 1]
    
    gate_1d = gate.arrayProject([0])  # [64]
    one_minus_gate_1d = gate_1d.multiply(-1).add(1)  # [64]
    
    f_fused_1d = f_sem_refined_1d.multiply(gate_1d).add(
                 f_phy_refined_1d.multiply(one_minus_gate_1d))  # [64]
    f_fused = f_fused_1d.toArray(1)  # [64, 1]
    
    w0, b0 = model_weights['classifier'][0]
    output = dnnLinear(f_fused, w0, b0, activate='relu')  # [32, 1]
    
    w1, b1 = model_weights['classifier'][1]
    output = dnnLinear(output, w1, b1, activate='relu')  # [16, 1]
    
    w2, b2 = model_weights['classifier'][2]
    output = dnnLinear(output, w2, b2, activate=None)  # [15, 1]
    
    logits_1d = output.arrayProject([0])  # [15]
    
    # 数值稳定的 Softmax
    logits_max = logits_1d.arrayReduce(ee.Reducer.max(), [0]).arrayGet([0])
    logits_shifted = logits_1d.subtract(logits_max)
    exp_logits = logits_shifted.exp()
    sum_exp = exp_logits.arrayReduce(ee.Reducer.sum(), [0]).arrayGet([0])
    probabilities = exp_logits.divide(sum_exp)  # [15]
    
    classLabel = probabilities.arrayArgmax().arrayGet([0])
    
    probMax = probabilities.arrayReduce(ee.Reducer.max(), [0]).arrayGet([0])
    
    # 计算语义流的平均门控权重（用于可视化分析）
    avg_sem_gate = gate_1d.arrayReduce(ee.Reducer.mean(), [0]).arrayGet([0])
    
    return classLabel, probMax, avg_sem_gate


# ==================== 推理 ====================

print("=" * 60)
print("开始改进版 GEE 推理...")
print("=" * 60)

# 标准化
print("[1/3] 标准化输入影像...")
img_standardized = standardizeImage(LL_clipped1, mean_list, std_list, featureName)


print("[2/3] 执行双流推理...")
classLabel, probMax, sem_gate_weight = improvedDualStreamInference(
    img_standardized, 
    model_weights, 
    split_idx=64, 
    n_features=94
)

# 重命名
classLabel = classLabel.rename('class')
probMax = probMax.rename('confidence')
sem_gate_weight = sem_gate_weight.rename('sem_weight')

print("[3/3] 推理完成!")
print("=" * 60)


visPalette = [
    '#BFBFBF','#F9F3C0','#3C8A5B','#A4CA73',
    '#007AFF','#C14646','#FFAA00','#00CC99',
    '#9900CC','#FF33CC','#33CCFF','#666600',
    '#CC6666','#0066CC',"#979731"
]

Map.addLayer(classLabel, {'palette': visPalette, 'min': 0, 'max': 14}, 'Classification_Improved', True)

# 显示
Map.addLayer(sem_gate_weight, {'min': 0.3, 'max': 0.7, 'palette': ['blue', 'white', 'red']}, 
             'Semantic_Gate_Weight', False)

Map

开始改进版 GEE 推理...
[1/3] 标准化输入影像...
[2/3] 执行双流推理...
[3/3] 推理完成!


Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(childr…

In [ ]:

from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, f1_score

model.eval()
y_true, y_pred = [], []

SPLIT_IDX = 64 

with torch.no_grad():
    for inputs, labels in valLoader:
        inputs = inputs.to(device)
        
        x_sem = inputs[:, :SPLIT_IDX]
        x_phy = inputs[:, SPLIT_IDX:]
        
        outputs, _ = model(x_sem, x_phy)
        _, preds = torch.max(outputs, 1)
        
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

oa = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred, average='weighted') 
report = classification_report(y_true, y_pred, digits=4)
cm = confusion_matrix(y_true, y_pred)

class_names = [str(i) for i in range(15)] 
cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)

print(f"Overall Accuracy (OA): {oa:.4f}")
print(f"Weighted F1 Score: {f1:.4f}\n")
print("Classification Report:\n", report)
print("\nConfusion Matrix:")
print(cm_df)

Overall Accuracy (OA): 0.9779
Weighted F1 Score: 0.9778

Classification Report:
               precision    recall  f1-score   support

         0.0     0.9863    1.0000    0.9931        72
         1.0     0.9792    1.0000    0.9895        47
         2.0     0.9811    1.0000    0.9905        52
         3.0     0.9610    0.9250    0.9427        80
         4.0     0.9796    1.0000    0.9897        48
         5.0     1.0000    0.9787    0.9892        47
         6.0     0.9808    0.9808    0.9808        52
         7.0     0.9206    0.9355    0.9280        62
         8.0     0.9753    0.9634    0.9693        82
         9.0     0.9839    0.9839    0.9839        62
        10.0     1.0000    1.0000    1.0000        48
        11.0     0.9833    1.0000    0.9916        59
        12.0     0.9762    0.9762    0.9762        42
        13.0     1.0000    0.9592    0.9792        49
        14.0     0.9825    1.0000    0.9912        56

    accuracy                         0.9779       858

In [ ]:
# 批处理导出到 Google Drive
task = ee.batch.Export.image.toDrive(
    image=classLabel,
    description='DNN_100_double_gaijin',          # Drive 上的文件夹名
    fileNamePrefix='classLabel1',   # 导出文件前缀
    scale=10,
    region=roi.geometry(),
    maxPixels=1e13
)
task.start()
print("已启动 Drive 导出任务，稍后在 Google Drive 的gee_exports文件夹中查看。")

已启动 Drive 导出任务，稍后在 Google Drive 的gee_exports文件夹中查看。
